# Notebook 2: Dataset Statistics & Cookbook Example 2

**Cookbook Example 2 — Gene-to-drug lookup and mechanism landscape analysis**

This notebook shows how to:
1. Load the validated dataset and query it programmatically
2. Build a gene → approved drug lookup (real-world use case)
3. Visualize mechanism class distribution, therapeutic area coverage,
   and approval-year trends across the 50-drug seed list

No API calls needed — everything runs from `data/validated/validated_triples.jsonl`.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

validated_path = Path('../data/validated/validated_triples.jsonl')
records = [
    json.loads(line)
    for line in validated_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
print(f'Loaded {len(records)} validated records.')

## Part A: Cookbook Example 2 — Gene-to-Drug Lookup

A frequent bioinformatics workflow: *"I have a patient with a pathogenic SMN1 variant —
what FDA-approved drugs target this gene?"*

In [ ]:
# ── Build gene → [drug, ...] index ────────────────────────────────────────────
from collections import defaultdict

gene_to_drugs: dict[str, list[dict]] = defaultdict(list)

for r in records:
    t = r['triple']
    for gene in t['associated_genes']:
        gene_to_drugs[gene].append({
            'generic':        t['drug_name_generic'],
            'brand':          t['drug_name_brand'],
            'mechanism_class': t['mechanism_class'],
            'approval_date':  t.get('approval_date', ''),
            'variant_context': t['variant_context'],
            'mechanism_summary': t['mechanism_summary'],
        })

# Sort each list by approval date
for gene in gene_to_drugs:
    gene_to_drugs[gene].sort(key=lambda x: x['approval_date'] or '9999')

print(f'Genes with approved drugs: {len(gene_to_drugs)}')
print('Genes:', sorted(gene_to_drugs.keys()))

In [ ]:
# ── Query: all drugs for SMN2 (SMA drugs that modulate SMN2 splicing) ─────────
query_gene = 'SMN2'
drugs = gene_to_drugs.get(query_gene, [])
print(f'FDA-approved drugs for patients with {query_gene}-relevant variants:')
print()
for d in drugs:
    print(f"  {d['generic']:30s} ({d['brand']})")
    print(f"    Mechanism:  {d['mechanism_class']} — {d['mechanism_summary'][:90]}…")
    print(f"    Approved:   {d['approval_date']}")
    print(f"    Variant:    {d['variant_context'][:80]}…")
    print()

In [ ]:
# ── Query: all siRNA drugs and their targets ──────────────────────────────────
sirna_drugs = [
    r['triple'] for r in records
    if r['triple']['mechanism_class'] == 'siRNA'
]
print(f"siRNA drugs in validated set ({len(sirna_drugs)}):")
print()
for t in sorted(sirna_drugs, key=lambda x: x['approval_date'] or '9999'):
    print(f"  {t['drug_name_generic']:20s} → target: {t['molecular_target'][:50]}")
    print(f"    Disease: {t['indicated_disease'][:60]}")

## Part B: Dataset Statistics

The following charts use the 10-record validated set.
They will auto-update as more records are added and validated.

In [ ]:
# ── Flatten to DataFrame ──────────────────────────────────────────────────────
rows = []
for r in records:
    t = r['triple']
    m = r['metadata']
    rows.append({
        'generic':          t['drug_name_generic'],
        'brand':            t['drug_name_brand'],
        'mechanism_class':  t['mechanism_class'],
        'approval_year':    int(t['approval_date'][:4]) if t.get('approval_date') and t['approval_date'] != 'investigational' else None,
        'n_genes':          len(t['associated_genes']),
        'chembl_validated': m['chembl_validated'],
        'drugbank_validated': m['drugbank_validated'],
        'confidence_score': r.get('confidence_score'),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── Plot 1: Mechanism class distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

mech_counts = df['mechanism_class'].value_counts()
axes[0].barh(mech_counts.index, mech_counts.values, color=sns.color_palette('tab10', len(mech_counts)))
axes[0].set_xlabel('Number of validated records')
axes[0].set_title('Mechanism class distribution\n(validated records, v0.1.0)', fontweight='bold')
for i, v in enumerate(mech_counts.values):
    axes[0].text(v + 0.05, i, str(v), va='center')

# ── Plot 2: Approval year trend ───────────────────────────────────────────────
year_counts = df.dropna(subset=['approval_year'])['approval_year'].value_counts().sort_index()
axes[1].bar(year_counts.index.astype(str), year_counts.values,
            color=sns.color_palette('Blues_d', len(year_counts)))
axes[1].set_xlabel('Approval year')
axes[1].set_ylabel('Count')
axes[1].set_title('FDA approval year\n(validated records, v0.1.0)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/validated/mechanism_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: data/validated/mechanism_distribution.png')

In [ ]:
# ── Plot 3: Validation coverage (ChEMBL + DrugBank) ──────────────────────────
val_df = pd.DataFrame({
    'Source': ['ChEMBL', 'DrugBank'],
    'Validated':    [df['chembl_validated'].sum(), df['drugbank_validated'].sum()],
    'Not validated': [len(df) - df['chembl_validated'].sum(),
                      len(df) - df['drugbank_validated'].sum()],
})

fig, ax = plt.subplots(figsize=(6, 4))
val_df.set_index('Source')[['Validated', 'Not validated']].plot(
    kind='bar', ax=ax, color=['#2ca02c', '#d62728'], edgecolor='white'
)
ax.set_title('Cross-validation coverage', fontweight='bold')
ax.set_ylabel('Records')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../data/validated/validation_coverage.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Seed-level analysis (all 50 seeds, not just validated) ───────────────────
seeds = json.loads(Path('../data/seeds/drug_seeds.json').read_text(encoding='utf-8'))
seeds_df = pd.DataFrame(seeds)

print(f'Total seed drugs: {len(seeds_df)}')
print()
print('Mechanism class distribution (seeds):')
print(seeds_df['mechanism_hint'].value_counts().to_string())
print()
print('Investigational:', seeds_df['is_investigational'].sum())
print('Needs manual verification:', seeds_df.get('needs_manual_verification', pd.Series([False]*len(seeds_df))).sum())

In [ ]:
# ── Seed mechanism class pie chart ────────────────────────────────────────────
seed_mech = seeds_df['mechanism_hint'].dropna().value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    seed_mech.values,
    labels=seed_mech.index,
    autopct=lambda p: f'{p:.1f}%\n({int(p * sum(seed_mech.values) / 100)})',
    startangle=140,
    colors=sns.color_palette('tab10', len(seed_mech)),
    pctdistance=0.75,
)
for text in autotexts:
    text.set_fontsize(9)
ax.set_title('Seed drug mechanism class distribution (n=50)', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../data/validated/seed_mechanism_pie.png', bbox_inches='tight')
plt.show()

## Part C: Drug-Repurposing Prior (Cookbook Example 2 continued)

A practical drug-repurposing pattern: find all mechanism classes used for a
given gene, then check if any target the same gene via a *different* mechanism.

In [ ]:
def repurposing_candidates(query_gene: str) -> pd.DataFrame:
    """Return all drugs hitting query_gene grouped by mechanism class."""
    drugs = gene_to_drugs.get(query_gene.upper(), [])
    if not drugs:
        print(f'No approved drugs found for {query_gene}')
        return pd.DataFrame()
    
    df_q = pd.DataFrame(drugs)
    print(f"Drugs targeting {query_gene} ({len(df_q)} found):")
    print(df_q[['generic', 'brand', 'mechanism_class', 'approval_date']].to_string(index=False))
    print()
    
    n_classes = df_q['mechanism_class'].nunique()
    if n_classes > 1:
        print(f'  → {n_classes} distinct mechanism classes hit this gene:')
        print(f'  → {list(df_q["mechanism_class"].unique())}')
        print('  → Multi-mechanism convergence suggests this gene is a high-value target.')
    return df_q

# TTR is the canonical example: 4 approved drugs (2 ASO, 1 siRNA x2, one siRNA)
repurposing_candidates('TTR')

In [ ]:
# GLA: enzyme replacement (agalsidase beta, pegunigalsidase) AND chaperone (migalastat)
repurposing_candidates('GLA')

In [ ]:
# SMN1/SMN2: gene therapy + ASO + small molecule — three mechanism classes for same gene
repurposing_candidates('SMN2')

## Summary

| Use case | Code pattern |
|---|---|
| Gene → drugs | `gene_to_drugs[gene]` |
| Mechanism filter | `[r for r in records if r['triple']['mechanism_class'] == 'siRNA']` |
| Approval trend | `df['approval_year'].value_counts().sort_index()` |
| Multi-mechanism convergence | `repurposing_candidates(gene)` |
| HF Dataset load | `load_from_disk('data/validated/hf_dataset')` |

For benchmark evaluation, the `id` field is stable across pipeline reruns —
use it as a join key against held-out LLM responses.